# Critical Node Detection in Aerial Swarms — Split, Preprocessing and Feature Engineering

## 1. Configuration

In [ ]:
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# paths
DATA_PATH = "swarm_criticality_dataset_100_scenarios.csv"
OUT_DIR = "processed"

os.makedirs(OUT_DIR, exist_ok=True)

# split settings
SEED = 42
VAL_FRACTION = 0.20
TEST_FRACTION = 0.20
SIZE_BINS = [0, 30, 52, 75]          # small: 18-30, medium: 37-52, large: 60-75 agents
SIZE_LABELS = ["small", "medium", "large"]
MAX_SPLIT_TRIES = 200                # redraws until val and test contain every attrition level
TRAIN_STEP_STRIDE = 1                # >1 keeps every k-th recorded step in the TRAIN split only

# constants from the README
SENSING_RANGE = 55.0
ATTRITION_ORDER = ["none", "light", "medium", "heavy"]
MAX_FPA = np.pi / 6                  # flight path angle limit (0.5236 rad)
MAX_HEADING_RATE = 0.65

# preprocessing settings (everything is learned on training rows only)
CORR_THRESHOLD = 0.95                # drop one feature of a pair if |Spearman| is above this
SKEW_THRESHOLD = 1.0                 # log1p for non-negative features with training skew above this
WINSOR_LOW = 0.001
WINSOR_HIGH = 0.999
STAT_SAMPLE = 300000                 # training rows used for correlation, VIF and screening

# parquet if pyarrow is installed, otherwise compressed csv
try:
    import pyarrow
    SAVE_FORMAT = "parquet"
except ImportError:
    SAVE_FORMAT = "csv.gz"
print("save format:", SAVE_FORMAT)

## 2. Load, type and validate

Only checks here. Nothing is learned from the data in this section.

In [ ]:
t0 = time.time()

if DATA_PATH.endswith(".parquet"):
    df = pd.read_parquet(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

TARGET = "is_critical"

required = ["case_id", "num_agents", "n_obstacles", "obstacle_density", "mean_obstacle_size", "clutter_index",
            "obstacle_clustering", "attrition_rate", "step", "agent_id", "rel_x", "rel_y", "rel_z",
            "dist_to_centroid", "rank_dist_to_centroid", "dist_to_target", "rank_dist_to_target",
            "vx", "vy", "vz", "speed", "heading", "flight_path_angle", "accel", "heading_rate",
            "num_neighbors", "mean_neighbor_dist", "min_neighbor_dist", "weighted_density", "angular_coverage",
            "criticality_score", "degree_centrality", "closeness_centrality", "betweenness_centrality",
            "eigenvector_centrality", "is_critical", "was_promoted"]

missing_cols = [c for c in required if c not in df.columns]
assert len(missing_cols) == 0, "missing columns: " + str(missing_cols)

# fix the data types
df[TARGET] = df[TARGET].astype("int8")
df["was_promoted"] = df["was_promoted"].astype("int8")
df["attrition_rate"] = pd.Categorical(df["attrition_rate"], categories=ATTRITION_ORDER, ordered=True)
assert df["attrition_rate"].notna().all(), "unknown attrition label"

df = df.sort_values(["case_id", "step", "agent_id"], kind="stable").reset_index(drop=True)

# drop rows with NaN or inf (the EDA found none)
num_cols = df.select_dtypes(include=np.number).columns
bad_rows = ~np.isfinite(df[num_cols].to_numpy(dtype="float64")).all(axis=1)
if bad_rows.any():
    print("dropping", bad_rows.sum(), "rows with NaN or inf")
    df = df[~bad_rows].reset_index(drop=True)

# every (case, step, agent) should appear only once
assert not df.duplicated(["case_id", "step", "agent_id"]).any(), "duplicate (case, step, agent) keys"

n_agents_total = df.groupby(["case_id", "agent_id"]).ngroups
print(f"loaded {len(df):,} rows x {df.shape[1]} columns, {df['case_id'].nunique()} cases, "
      f"{n_agents_total:,} agents in {time.time() - t0:.0f}s")
print("critical share:", round(df[TARGET].mean(), 4))

## 3. Column roles

Every column gets exactly one role. The assertions at the end of the notebook check that no leaking column ends up in a feature list.

| Role | Columns | Treatment |
| --- | --- | --- |
| Metadata | `case_id`, `step`, `agent_id` | Used for splitting, grouping and analysis. **Never features.** |
| Target | `is_critical` | Target only |
| Excluded (leakage) | `criticality_score`, `was_promoted` | Saved with an `aux_` prefix for the heuristic baseline and label analysis. **Never features.** |
| Environment | `num_agents`, `n_obstacles`, `obstacle_density`, `mean_obstacle_size`, `clutter_index`, `obstacle_clustering`, `attrition_rate` | Constant inside a case. Only used in FS_C. Also saved with a `strat_` prefix for robustness analysis |
| Agent / swarm raw | everything else | Candidate features |

In [ ]:
META_COLS = ["case_id", "step", "agent_id"]
LEAK_COLS = ["criticality_score", "was_promoted"]
ENV_NUM = ["num_agents", "n_obstacles", "obstacle_density", "mean_obstacle_size", "clutter_index", "obstacle_clustering"]
ENV_CAT = ["attrition_rate"]

# vx, vy, vz can be rebuilt from speed, heading and flight_path_angle, so they are not used
# heading is an angle (circular), so it is replaced by sin and cos later
DROPPED_RAW = ["vx", "vy", "vz", "heading"]

RAW_AGENT = ["rel_x", "rel_y", "rel_z", "dist_to_centroid", "rank_dist_to_centroid", "dist_to_target",
             "rank_dist_to_target", "speed", "flight_path_angle", "accel", "heading_rate",
             "num_neighbors", "mean_neighbor_dist", "min_neighbor_dist", "weighted_density", "angular_coverage",
             "degree_centrality", "closeness_centrality", "betweenness_centrality", "eigenvector_centrality"]

roles = {"metadata": META_COLS, "target": [TARGET], "leakage": LEAK_COLS,
         "environment": ENV_NUM + ENV_CAT, "raw agent features": RAW_AGENT, "dropped raw": DROPPED_RAW}

# check that every column has exactly one role
assigned = []
for role in roles:
    assigned = assigned + roles[role]
assert len(assigned) == len(set(assigned)), "a column has two roles"
assert set(assigned) == set(required), "unassigned columns: " + str(set(required) - set(assigned))

for role in roles:
    print(role, ":", len(roles[role]), "columns")

## 4. Case-level 60-20-20 split

**The unit of splitting is `case_id`.** All rows of a scenario (every timestep and every agent) go to exactly one split.

**Stratification.** There are only 100 cases, so a plain random split could give a validation set without any heavy-attrition large swarm. Cases are grouped by swarm-size bucket and attrition level (12 groups). Inside each group the cases are shuffled, about 20% go to validation, about 20% to test and the rest to train. If a draw leaves validation or test without some attrition level, the seed is increased and the draw is repeated. Model performance is never used to choose the split.

In [ ]:
# one row per case
case_tbl = df.groupby("case_id", observed=True).agg(
    num_agents=("num_agents", "first"),
    attrition=("attrition_rate", "first"),
    n_obstacles=("n_obstacles", "first"),
    obstacle_density=("obstacle_density", "first"),
    mean_obstacle_size=("mean_obstacle_size", "first"),
    clutter_index=("clutter_index", "first"),
    obstacle_clustering=("obstacle_clustering", "first"),
    rows=("agent_id", "size"),
    critical_rows=(TARGET, "sum"),
)
case_tbl["size_bucket"] = pd.cut(case_tbl["num_agents"], SIZE_BINS, labels=SIZE_LABELS)
N = len(case_tbl)

print("cases per group (size bucket x attrition):")
print(pd.crosstab(case_tbl["size_bucket"], case_tbl["attrition"], margins=True))

# the 12 groups: for each (size bucket, attrition) the positions of its cases in case_tbl
strata = case_tbl.groupby(["size_bucket", "attrition"], observed=True).indices


def stratified_case_split(seed):
    """Returns an array with one number per case: 0 = train, 1 = val, 2 = test."""
    rng = np.random.default_rng(seed)
    assign = np.zeros(N, dtype=np.int8)
    for key in strata:
        idx = rng.permutation(strata[key])
        n = len(idx)
        n_val = min(int(round(n * VAL_FRACTION)), max(n - 1, 0))
        n_test = min(int(round(n * TEST_FRACTION)), max(n - n_val - 1, 0))
        assign[idx[:n_val]] = 1
        assign[idx[n_val:n_val + n_test]] = 2
    return assign


def has_all_attrition_levels(assign):
    """True if both val and test contain every attrition level."""
    attrition = case_tbl["attrition"].to_numpy()
    for s in [1, 2]:
        if set(attrition[assign == s]) != set(ATTRITION_ORDER):
            return False
    return True


split_seed = None
split_code = None
for k in range(MAX_SPLIT_TRIES):
    seed_try = SEED + k
    assign = stratified_case_split(seed_try)
    if has_all_attrition_levels(assign):
        split_seed = seed_try
        split_code = assign
        break

assert split_code is not None, "no draw covered every attrition level; increase MAX_SPLIT_TRIES"
print("accepted draw: seed", split_seed, "(draw", k + 1, "of up to", MAX_SPLIT_TRIES, ")")

case_split = pd.Series(np.array(["train", "val", "test"])[split_code], index=case_tbl.index, name="split")
case_split.to_csv(OUT_DIR + "/case_split.csv")

In [ ]:
df["split"] = df["case_id"].map(case_split)
assert df["split"].notna().all()
case_tbl["split"] = case_split
order = ["train", "val", "test"]

# summary of the three splits
summary = df.groupby("split").agg(cases=("case_id", "nunique"), rows=("agent_id", "size"), critical_share=(TARGET, "mean"))
summary["agents"] = df.drop_duplicates(["case_id", "agent_id"]).groupby("split").size()
summary["row_share"] = summary["rows"] / summary["rows"].sum()
print(summary.loc[order])

print("\ncases by split and attrition:")
print(pd.crosstab(case_tbl["split"], case_tbl["attrition"]).loc[order])

print("\ncases by split and swarm-size bucket:")
print(pd.crosstab(case_tbl["split"], case_tbl["size_bucket"]).loc[order])

print("\nrow share by attrition inside each split:")
print(pd.crosstab(df["split"], df["attrition_rate"], normalize="index").loc[order])

print("\nmean environment values by split:")
print(case_tbl.groupby("split")[["num_agents", "n_obstacles", "obstacle_density", "clutter_index", "obstacle_clustering"]].mean().loc[order])

# hard checks
train_cases = set(df[df["split"] == "train"]["case_id"])
val_cases = set(df[df["split"] == "val"]["case_id"])
test_cases = set(df[df["split"] == "test"]["case_id"])
assert train_cases.isdisjoint(val_cases)
assert train_cases.isdisjoint(test_cases)
assert val_cases.isdisjoint(test_cases)
assert len(train_cases) + len(val_cases) + len(test_cases) == df["case_id"].nunique()
assert df.groupby(["case_id", "step"], observed=True)["split"].nunique().max() == 1, "a snapshot is split across sets"
print("\nsplit checks passed: cases are disjoint, snapshots are whole, every attrition level is in val and test")

## 5. Edge cases and flags

* An agent with no neighbours has no neighbour distances. They are set to the sensing range (55) and the agent is flagged as isolated.
* Flags are added for agents that are at their turn-rate limit or climb-angle limit.

In [ ]:
isolated = df["num_neighbors"] == 0
df["is_isolated"] = isolated.astype("int8")
df.loc[isolated, "mean_neighbor_dist"] = SENSING_RANGE
df.loc[isolated, "min_neighbor_dist"] = SENSING_RANGE

df["turn_saturated"] = (df["heading_rate"].abs() >= MAX_HEADING_RATE - 1e-6).astype("int8")
df["pitch_saturated"] = (df["flight_path_angle"].abs() >= MAX_FPA - 1e-6).astype("int8")

print("isolated rows:", int(isolated.sum()), "(", round(isolated.mean() * 100, 3), "%)")
print("turn saturated:", round(df["turn_saturated"].mean() * 100, 1), "%")
print("pitch saturated:", round(df["pitch_saturated"].mean() * 100, 1), "%")

## 6. Feature engineering

Every feature is computed from the agent's own row and, for swarm-context features, from the other rows of the **same (case, step) snapshot**. No feature looks at another timestep, another case or a label, so the same function works unchanged on a new swarm and cannot leak across the split. This is tested in section 11.

In [ ]:
KEYS = ["case_id", "step"]

# columns that get a "z-score inside the snapshot" and a "percentile rank inside the snapshot"
Z_COLS = ["num_neighbors", "mean_neighbor_dist", "min_neighbor_dist", "weighted_density", "angular_coverage",
          "degree_centrality", "closeness_centrality", "betweenness_centrality", "eigenvector_centrality",
          "dist_to_centroid", "speed"]
PCT_COLS = ["num_neighbors", "min_neighbor_dist", "weighted_density", "angular_coverage", "eigenvector_centrality"]

# features that use only the agent's own row
FE_LOCAL = ["heading_sin", "heading_cos", "rel_azimuth_sin", "rel_azimuth_cos", "rel_elevation", "radial_speed",
            "radial_speed_norm", "nn_dist_ratio", "is_isolated", "turn_saturated", "pitch_saturated"]

# features that also use the other agents of the same snapshot
FE_CONTEXT = ["n_alive", "alive_frac", "n_lost", "swarm_radius", "swarm_mean_dist", "swarm_mean_degree",
              "swarm_frac_isolated", "dist_centroid_rel_radius", "dist_centroid_rel_mean",
              "rel_x_norm", "rel_y_norm", "rel_z_norm", "dist_target_rel_radius", "neighbor_frac",
              "rel_speed_to_swarm", "heading_dev_abs"]
FE_CONTEXT = FE_CONTEXT + [c + "_z" for c in Z_COLS] + [c + "_pct" for c in PCT_COLS]


def engineer_features(d):
    """Takes a DataFrame of rows and returns a DataFrame of engineered columns with the same index."""
    d = d.copy()
    fe = pd.DataFrame(index=d.index)

    # ----- features from the agent's own row
    fe["heading_sin"] = np.sin(d["heading"])
    fe["heading_cos"] = np.cos(d["heading"])

    rho = np.sqrt(d["rel_x"] ** 2 + d["rel_y"] ** 2 + d["rel_z"] ** 2).clip(lower=1e-9)
    azimuth = np.arctan2(d["rel_y"], d["rel_x"])
    fe["rel_azimuth_sin"] = np.sin(azimuth)
    fe["rel_azimuth_cos"] = np.cos(azimuth)
    fe["rel_elevation"] = np.arcsin((d["rel_z"] / rho).clip(-1, 1))

    # how fast the agent moves away from the swarm centre
    fe["radial_speed"] = (d["rel_x"] * d["vx"] + d["rel_y"] * d["vy"] + d["rel_z"] * d["vz"]) / rho
    fe["radial_speed_norm"] = fe["radial_speed"] / d["speed"].clip(lower=1e-9)

    fe["nn_dist_ratio"] = (d["min_neighbor_dist"] / d["mean_neighbor_dist"].clip(lower=1e-9)).clip(0, 1)
    fe["is_isolated"] = d["is_isolated"]
    fe["turn_saturated"] = d["turn_saturated"]
    fe["pitch_saturated"] = d["pitch_saturated"]

    # ----- swarm state of this snapshot
    g = d.groupby(KEYS)
    n_alive = g["agent_id"].transform("size")
    fe["n_alive"] = n_alive
    fe["alive_frac"] = n_alive / d["num_agents"]
    fe["n_lost"] = d["num_agents"] - n_alive

    radius = g["dist_to_centroid"].transform("max").clip(lower=1e-9)
    mean_dist = g["dist_to_centroid"].transform("mean").clip(lower=1e-9)
    fe["swarm_radius"] = radius
    fe["swarm_mean_dist"] = mean_dist
    fe["swarm_mean_degree"] = g["degree_centrality"].transform("mean")
    fe["swarm_frac_isolated"] = g["is_isolated"].transform("mean")

    fe["dist_centroid_rel_radius"] = d["dist_to_centroid"] / radius
    fe["dist_centroid_rel_mean"] = d["dist_to_centroid"] / mean_dist
    fe["rel_x_norm"] = d["rel_x"] / radius
    fe["rel_y_norm"] = d["rel_y"] / radius
    fe["rel_z_norm"] = d["rel_z"] / radius
    fe["dist_target_rel_radius"] = (d["dist_to_target"] - g["dist_to_target"].transform("mean")) / radius
    fe["neighbor_frac"] = d["num_neighbors"] / (n_alive - 1).clip(lower=1)

    # ----- motion compared with the rest of the swarm
    dv_squared = 0
    for v in ["vx", "vy", "vz"]:
        dv_squared = dv_squared + (d[v] - g[v].transform("mean")) ** 2
    fe["rel_speed_to_swarm"] = np.sqrt(dv_squared)

    # average heading of the snapshot (headings are angles, so average sin and cos)
    d["_sin"] = np.sin(d["heading"])
    d["_cos"] = np.cos(d["heading"])
    mean_sin = d.groupby(KEYS)["_sin"].transform("mean")
    mean_cos = d.groupby(KEYS)["_cos"].transform("mean")
    mean_heading = np.arctan2(mean_sin, mean_cos)
    heading_diff = d["heading"] - mean_heading
    fe["heading_dev_abs"] = np.abs(np.arctan2(np.sin(heading_diff), np.cos(heading_diff)))

    # ----- standing inside the swarm: z-score and percentile rank within the snapshot
    for c in Z_COLS:
        centred = d[c] - g[c].transform("mean")
        d["_sq"] = centred ** 2
        std = np.sqrt(d.groupby(KEYS)["_sq"].transform("mean"))
        z = centred / std.where(std > 1e-12, np.nan)
        fe[c + "_z"] = z.fillna(0.0)

    for c in PCT_COLS:
        fe[c + "_pct"] = g[c].rank(pct=True, method="average")

    fe = fe[FE_LOCAL + FE_CONTEXT]
    return fe.astype("float32")


t0 = time.time()
fe = engineer_features(df)
assert np.isfinite(fe.to_numpy()).all(), "engineered features contain NaN or inf"

# attrition as a number (0 none ... 3 heavy) and as one-hot columns
fe["attrition_level"] = df["attrition_rate"].cat.codes.astype("float32")
ATTR_ONEHOT = []
for a in ATTRITION_ORDER:
    fe["attr_" + a] = (df["attrition_rate"] == a).astype("float32")
    ATTR_ONEHOT.append("attr_" + a)

print(fe.shape[1], "engineered columns in", round(time.time() - t0), "s")
fe.describe().T[["mean", "std", "min", "max"]]

## 7. Assemble the three feature sets

Raw and engineered features are joined into one table. The three sets are nested:

* **FS_A**: raw agent features + features from the agent's own row
* **FS_B**: FS_A + swarm-context features
* **FS_C**: FS_B + environment variables

The order of this list is also the priority order when two correlated features compete in section 8 (raw features win).

In [ ]:
X_all = pd.concat([df[RAW_AGENT + ENV_NUM].astype("float32"), fe], axis=1)

FS_A = RAW_AGENT + FE_LOCAL
FS_B = FS_A + FE_CONTEXT
FS_C = FS_B + ENV_NUM + ["attrition_level"]
print("FS_A:", len(FS_A), " FS_B:", len(FS_B), " FS_C:", len(FS_C))

# training rows only, and a random sample of them for the statistics below
train_idx = np.flatnonzero((df["split"] == "train").to_numpy())
rng = np.random.default_rng(SEED)
stat_idx = rng.choice(train_idx, size=min(STAT_SAMPLE, len(train_idx)), replace=False)
X_stat = X_all.iloc[stat_idx]
y_stat = df[TARGET].to_numpy()[stat_idx]

## 8. Redundancy pruning (training cases only)

1. Features with a single value in the training rows are dropped.
2. For every pair with |Spearman| above `CORR_THRESHOLD`, the feature that comes **later** in `FS_C` is dropped, so raw features win over engineered copies.
3. The VIF of what remains is printed for information only. Nothing is dropped because of it.

Dropped features are removed from all three sets, so FS_A ⊂ FS_B ⊂ FS_C still holds.

In [ ]:
feats = list(FS_C)
dropped = []        # rows of (feature, reason, kept_instead, abs_spearman)

# step 1: constant features
for c in list(feats):
    if X_stat[c].nunique() <= 1:
        dropped.append((c, "zero variance in training", "", np.nan))
        feats.remove(c)

# step 2: highly correlated pairs
corr = X_stat[feats].corr(method="spearman").abs()
position = {}
for i in range(len(feats)):
    position[feats[i]] = i

pairs = []
for i in range(len(feats)):
    for j in range(i + 1, len(feats)):
        r = corr.iloc[i, j]
        if r > CORR_THRESHOLD:
            pairs.append((feats[i], feats[j], r))
pairs.sort(key=lambda p: -p[2])        # strongest correlation first

gone = set()
for a, b, r in pairs:
    if a in gone or b in gone:
        continue
    # a comes before b in the list, so a is kept and b is dropped
    gone.add(b)
    dropped.append((b, "|Spearman| > " + str(CORR_THRESHOLD), a, r))

kept_C = [c for c in feats if c not in gone]
dropped_tbl = pd.DataFrame(dropped, columns=["feature", "reason", "kept_instead", "abs_spearman"])

FS_A_final = [c for c in FS_A if c in kept_C]
FS_B_final = [c for c in FS_B if c in kept_C]
FS_C_final = [c for c in FS_C if c in kept_C]
assert set(FS_A_final) <= set(FS_B_final) <= set(FS_C_final)
print("FS_A:", len(FS_A_final), " FS_B:", len(FS_B_final), " FS_C:", len(FS_C_final))
print("\ndropped features:")
dropped_tbl

In [ ]:
# VIF of the remaining features (information only)
Z = X_stat[FS_C_final].astype("float64")
Z = (Z - Z.mean()) / Z.std().replace(0, 1)
R = np.corrcoef(Z.to_numpy().T)
vif_final = pd.Series(np.diag(np.linalg.pinv(R)), index=FS_C_final, name="VIF").sort_values(ascending=False)
vif_final.head(15).to_frame()

In [ ]:
cm = X_stat[FS_C_final].corr(method="spearman")

plt.figure(figsize=(13, 11))
plt.imshow(cm, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
plt.xticks(range(len(cm)), cm.columns, rotation=90, fontsize=7)
plt.yticks(range(len(cm)), cm.columns, fontsize=7)
plt.colorbar(label="Spearman")
plt.title("FS_C after pruning (training rows)")
plt.tight_layout()
plt.show()

## 9. Preprocessing for neural / GNN inputs (training cases only)

Tree models (Random Forest, XGBoost, LightGBM) do not care about monotone transforms, so they use the **unscaled** table. GraphSAGE and any linear or MLP model use the **scaled** table. Everything below is learned from training rows only:

1. `log1p` for non-negative continuous features whose training skew is above `SKEW_THRESHOLD`
2. clipping at the training quantiles `WINSOR_LOW` and `WINSOR_HIGH` (after the log)
3. standardising with the training mean and standard deviation (0/1 flags are left unchanged)

Validation and test are only transformed, never used to fit.

In [ ]:
def fit_preprocessor(X, features):
    """Learns the log columns, clipping bounds, mean and std from X (training rows)."""
    X = X[features].astype("float64")

    # columns that only contain 0 and 1 are left alone
    binary_cols = []
    for c in features:
        if X[c].isin([0.0, 1.0]).all():
            binary_cols.append(c)
    cont_cols = [c for c in features if c not in binary_cols]

    skew = X[cont_cols].skew()
    log_cols = []
    for c in cont_cols:
        if X[c].min() >= 0 and skew[c] > SKEW_THRESHOLD:
            log_cols.append(c)

    Xl = X[cont_cols].copy()
    for c in log_cols:
        Xl[c] = np.log1p(Xl[c].clip(lower=0))

    low = Xl.quantile(WINSOR_LOW)
    high = Xl.quantile(WINSOR_HIGH)
    Xw = Xl.clip(low, high, axis=1)
    mean = Xw.mean()
    std = Xw.std().replace(0, 1.0)

    return {"features": list(features), "binary_cols": binary_cols, "cont_cols": cont_cols,
            "log_cols": log_cols, "low": low, "high": high, "mean": mean, "std": std}


def apply_preprocessor(X, pre):
    """Applies what was learned in fit_preprocessor. Works for train, val and test."""
    X = X[pre["features"]].astype("float64")
    Xc = X[pre["cont_cols"]].copy()
    for c in pre["log_cols"]:
        Xc[c] = np.log1p(Xc[c].clip(lower=0))
    Xc = Xc.clip(pre["low"], pre["high"], axis=1)
    Xc = (Xc - pre["mean"]) / pre["std"]
    out = pd.concat([Xc, X[pre["binary_cols"]]], axis=1)[pre["features"]]
    return out.astype("float32")


pre = fit_preprocessor(X_all.iloc[train_idx], FS_C_final)
joblib.dump(pre, OUT_DIR + "/preprocessor.joblib")
print("log1p applied to:", pre["log_cols"])
print("binary (left as 0/1):", pre["binary_cols"])

In [ ]:
# before / after of the log transform for a few skewed features
cols = pre["log_cols"][:6]
if len(cols) > 0:
    fig, axes = plt.subplots(2, len(cols), figsize=(3.6 * len(cols), 6), squeeze=False)
    sample = X_all.iloc[stat_idx]
    for j in range(len(cols)):
        c = cols[j]
        axes[0, j].hist(sample[c], bins=50)
        axes[0, j].set_title(c + "\nraw (skew " + str(round(sample[c].skew(), 2)) + ")", fontsize=9)
        logged = np.log1p(sample[c].clip(lower=0))
        axes[1, j].hist(logged, bins=50)
        axes[1, j].set_title("log1p (skew " + str(round(logged.skew(), 2)) + ")", fontsize=9)
    plt.tight_layout()
    plt.show()

## 10. Screening and leakage check (training rows only)

Every final feature is scored on its own against the target. **A single feature that separates the classes almost perfectly is treated as a leak and stops the notebook.** For comparison, `criticality_score` alone reaches ROC-AUC 0.99.

In [ ]:
rows = []
for c in FS_C_final:
    x = X_stat[c].to_numpy(dtype=float)
    if x.max() == x.min():
        continue
    auc = roc_auc_score(y_stat, x)
    if auc >= 0.5:
        sign = 1
    else:
        sign = -1

    if c in RAW_AGENT:
        group = "raw"
    elif c in ENV_NUM + ["attrition_level"]:
        group = "environment"
    elif c in FE_CONTEXT:
        group = "engineered-context"
    else:
        group = "engineered-local"

    rows.append({"feature": c, "group": group,
                 "roc_auc": max(auc, 1 - auc),
                 "pr_auc": average_precision_score(y_stat, sign * x)})

screen = pd.DataFrame(rows).set_index("feature").sort_values("pr_auc", ascending=False)
screen.to_csv(OUT_DIR + "/feature_screening_train.csv")

print("prevalence (PR-AUC of a random score) =", round(y_stat.mean(), 3))
print("\ntop 25 features:")
print(screen.head(25))
print("\nbest feature per group:")
print(screen.groupby("group")[["roc_auc", "pr_auc"]].max())

LEAK_LIMIT = 0.97
suspects = screen.index[screen["roc_auc"] > LEAK_LIMIT].tolist()
assert len(suspects) == 0, "single-feature ROC-AUC above " + str(LEAK_LIMIT) + ": " + str(suspects)
print("\nleak check passed: no single feature has ROC-AUC above", LEAK_LIMIT)

## 11. Integrity tests

These tests check the guarantees claimed above. They are cheap, so keep them in the notebook.

In [ ]:
# test 1: no metadata, target or leakage column in any feature set
forbidden = set(META_COLS) | {TARGET} | set(LEAK_COLS)
for name, fs in [("FS_A", FS_A_final), ("FS_B", FS_B_final), ("FS_C", FS_C_final)]:
    bad = set(fs) & forbidden
    assert len(bad) == 0, name + " contains a forbidden column: " + str(bad)
    for c in fs:
        assert not c.startswith("aux_") and not c.startswith("strat_")
print("1. no metadata, target or leakage column in any feature set")

# test 2: all final features are finite
assert np.isfinite(X_all[FS_C_final].to_numpy()).all()
print("2. all final features are finite")

# test 3: features of a snapshot do not depend on other cases or on row order
rng_t = np.random.default_rng(0)
case_ids = df["case_id"].unique()
test_cases_ids = rng_t.choice(case_ids, size=min(5, len(case_ids)), replace=False)
for cid in test_cases_ids:
    sub = df[df["case_id"] == cid]
    shuffled = sub.sample(frac=1.0, random_state=1)
    alone = engineer_features(shuffled).loc[sub.index]
    pd.testing.assert_frame_equal(alone, fe.loc[sub.index, FE_LOCAL + FE_CONTEXT], rtol=1e-4, atol=1e-5, check_dtype=False)
print("3. engineered features are identical when one case is processed alone with shuffled rows")

# test 4: the split can be rebuilt from its seed
assert (stratified_case_split(split_seed) == split_code).all()
print("4. split is reproducible from seed", split_seed)

# test 5: training data is standardised, val is allowed to differ
Xs_train = apply_preprocessor(X_all.iloc[train_idx], pre)
Xs_val = apply_preprocessor(X_all[(df["split"] == "val").to_numpy()], pre)
cont = pre["cont_cols"]
train_mean = Xs_train[cont].mean().abs().max()
val_mean = Xs_val[cont].mean().abs().max()
assert train_mean < 0.05, train_mean
print("5. train |mean| max", round(train_mean, 4), "| val |mean| max", round(val_mean, 4),
      "(val can differ because it was not used to fit)")
del Xs_train, Xs_val

## 12. Save the datasets

For each split two tables with the **same rows in the same order** are saved:

* `*_tree`: unscaled features for Random Forest / XGBoost / LightGBM
* `*_nn`: scaled features for GraphSAGE / MLP

Column prefixes make misuse hard: `aux_*` columns (the excluded `criticality_score` and `was_promoted`) and `strat_*` columns must never be used as model features. `ts_id` identifies a (case, step) snapshot: group by it for Precision@K / Recall@K and to build one graph per snapshot.

`TRAIN_STEP_STRIDE` thins the training split by whole snapshots only.

In [ ]:
meta = pd.DataFrame({
    "case_id": df["case_id"].astype(str),
    "step": df["step"],
    "agent_id": df["agent_id"],
    "ts_id": df.groupby(KEYS, sort=False).ngroup().astype("int32"),
    "split": df["split"],
    TARGET: df[TARGET],
    "aux_criticality_score": df["criticality_score"].astype("float32"),
    "aux_was_promoted": df["was_promoted"],
    "strat_num_agents": df["num_agents"],
    "strat_attrition": df["attrition_rate"].astype(str),
    "strat_n_obstacles": df["n_obstacles"],
    "strat_obstacle_density": df["obstacle_density"].astype("float32"),
    "strat_clutter_index": df["clutter_index"].astype("float32"),
})

# optionally thin the training split (whole snapshots only)
if TRAIN_STEP_STRIDE > 1:
    keep = (df["split"] != "train") | (((df["step"] // 2) % TRAIN_STEP_STRIDE) == 0)
else:
    keep = pd.Series(True, index=df.index)
print("rows kept:", int(keep.sum()), "of", len(df))


def write_table(table, name):
    path = OUT_DIR + "/" + name + "." + SAVE_FORMAT
    if SAVE_FORMAT == "parquet":
        table.to_parquet(path, index=False)
    else:
        table.to_csv(path, index=False)
    return path


written = {}
for s in ["train", "val", "test"]:
    m = ((df["split"] == s) & keep).to_numpy()

    tree = pd.concat([meta[m].reset_index(drop=True),
                      X_all.loc[m, FS_C_final].reset_index(drop=True)], axis=1)

    nn = pd.concat([meta.loc[m, ["case_id", "step", "agent_id", "ts_id", "split", TARGET]].reset_index(drop=True),
                    apply_preprocessor(X_all.loc[m], pre).reset_index(drop=True)], axis=1)

    p1 = write_table(tree, s + "_tree")
    p2 = write_table(nn, s + "_nn")
    written[s] = (p1, p2, len(tree))

In [ ]:
# feature lists, class weights and a short preprocessing report
feature_sets = {"FS_A": FS_A_final, "FS_B": FS_B_final, "FS_C": FS_C_final,
                "attrition_onehot_alternative": ATTR_ONEHOT,
                "target": TARGET,
                "never_features_prefixes": ["aux_", "strat_"],
                "metadata": ["case_id", "step", "agent_id", "ts_id", "split"]}
json.dump(feature_sets, open(OUT_DIR + "/feature_sets.json", "w"), indent=2)

train_meta = meta[(df["split"] == "train") & keep]
n_pos = int(train_meta[TARGET].sum())
n_neg = int(len(train_meta) - n_pos)

class_info = {}
for s in ["train", "val", "test"]:
    rows_s = (df["split"] == s) & keep
    class_info[s] = {"rows": int(rows_s.sum()), "critical_share": float(df.loc[rows_s, TARGET].mean())}
class_info["train_class_weights"] = {
    "n_positive": n_pos,
    "n_negative": n_neg,
    "scale_pos_weight": n_neg / max(n_pos, 1),
    "sklearn_class_weight": {0: 0.5 * len(train_meta) / max(n_neg, 1), 1: 0.5 * len(train_meta) / max(n_pos, 1)},
}
json.dump(class_info, open(OUT_DIR + "/class_info.json", "w"), indent=2)

report = {
    "data_path": DATA_PATH,
    "seed": SEED,
    "split_seed": int(split_seed),
    "n_cases": int(N),
    "corr_threshold": CORR_THRESHOLD,
    "dropped_features": dropped_tbl.fillna("").to_dict("records"),
    "log1p_columns": pre["log_cols"],
    "binary_columns": pre["binary_cols"],
    "winsor_quantiles": [WINSOR_LOW, WINSOR_HIGH],
    "winsor_low": {c: float(pre["low"][c]) for c in pre["cont_cols"]},
    "winsor_high": {c: float(pre["high"][c]) for c in pre["cont_cols"]},
    "mean": {c: float(pre["mean"][c]) for c in pre["cont_cols"]},
    "std": {c: float(pre["std"][c]) for c in pre["cont_cols"]},
    "train_step_stride": TRAIN_STEP_STRIDE,
}
json.dump(report, open(OUT_DIR + "/preprocessing_report.json", "w"), indent=2, default=str)

print("saved to folder:", OUT_DIR)
for s in written:
    p1, p2, n = written[s]
    print(" ", s, n, "rows ->", p1, "and", p2)
print("  also: feature_sets.json, class_info.json, preprocessing_report.json, preprocessor.joblib, case_split.csv, feature_screening_train.csv")